<a href="https://colab.research.google.com/github/DivyaSwamy/transformers_tutorials/blob/main/create_demo_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Author:- Divya Swaminathan

#### September 2025

#### <u>Goal</u> -
1) From the *"Aliounethegoat/classification-medicale-multi-cancer"*  cancer images dataset, create a dataset of only breat cancer images.
2) Push dataset to hub and now it is ready for image based analysis tasks.

#### Outcome -
  Hurray !!

In [2]:
import io
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import notebook_login

from datasets import load_dataset, DatasetDict

In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('huggingface_token') # Retrieve the token from secrets

if HF_TOKEN:
  login(HF_TOKEN)
  print("Successfully logged in to Hugging Face!")
else:
  print("Hugging Face token not found in Colab Secrets.")



Successfully logged in to Hugging Face!


In [4]:
path_to_dataset = "Aliounethegoat/classification-medicale-multi-cancer"

ds = load_dataset(path_to_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/367 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

data/train-00000-of-00021.parquet:   0%|          | 0.00/643M [00:00<?, ?B/s]

...

data/train-00018-of-00021.parquet:   0%|          | 0.00/346M [00:00<?, ?B/s]

data/train-00019-of-00021.parquet:   0%|          | 0.00/229M [00:00<?, ?B/s]

data/train-00020-of-00021.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130002 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

#### Isolate all breast cancer images from ds.

There are many different kinds of images in *ds*.
I know which label to pick for breast cancer after looking at *label-names* in ds

In [5]:
# Isolate all breast cancer images from ds.
breast_cancer = ds.filter(lambda x: x['label_name'] == 'cancer_sein')

Filter:   0%|          | 0/130002 [00:00<?, ? examples/s]

#### Save the breast cancer dataset.

In [7]:
breast_cancer

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'label_name'],
        num_rows: 10000
    })
})

#### Generate train, test and validate splits for breast_cancer dataset

In [8]:
split_dataset = breast_cancer['train'].train_test_split(test_size=0.25) # 25% test, 75% train
validation_dataset = split_dataset['test'].train_test_split(test_size=0.5) # 50% validation, 50% test

In [9]:
final_dataset = DatasetDict({
    'train': split_dataset['train'],
    'validation': validation_dataset['train'],
    'test': validation_dataset['test']
})

In [10]:
final_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'label_name'],
        num_rows: 7500
    })
    validation: Dataset({
        features: ['image', 'label', 'label_name'],
        num_rows: 1250
    })
    test: Dataset({
        features: ['image', 'label', 'label_name'],
        num_rows: 1250
    })
})

#### Check if dataset is balanced.


In [11]:
# Check if the dataset is balanced
from collections import defaultdict

def count_labels(dataset):
  """
  """
  count_label = defaultdict(int)

  for label in dataset['label']:
    count_label[label] += 1

  return count_label

In [12]:
for idx in ['train', 'test', 'validation']:
  print(idx, count_labels(final_dataset[idx]))


train defaultdict(<class 'int'>, {'breast_malignant': 3776, 'breast_benign': 3724})
test defaultdict(<class 'int'>, {'breast_benign': 641, 'breast_malignant': 609})
validation defaultdict(<class 'int'>, {'breast_malignant': 615, 'breast_benign': 635})


#### Push to hub.

Dataset is now ready to play with.

In [ ]:
final_dataset.push_to_hub("divyaswamy/breast_cancer_demo")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/3750 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/38 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   1%|1         | 1.40MB /  124MB            

CommitInfo(commit_url='https://huggingface.co/datasets/divyaswamy/breast_cancer_demo/commit/c1fd179d91fadda4569493eca26df8fe449cac0d', commit_message='Upload dataset', commit_description='', oid='c1fd179d91fadda4569493eca26df8fe449cac0d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/divyaswamy/breast_cancer_demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='divyaswamy/breast_cancer_demo'), pr_revision=None, pr_num=None)